# CatBoost Training

CatBoost or Categorical Boosting was developed by Yandex.

It is an ML algorithm based on gradient-boosted decision trees. It supports categorical values natively, handles null values automatically, and uses ordered boosting to prevent data leakage.

## Import Libraries

Import the required libraries.

In [ ]:
# Import required libraries
import os
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, classification_report, confusion_matrix,
    precision_recall_curve
)
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier

# hide optuna log output but let the warnings show
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Data Loading

Taking the input data

In [ ]:
# Set paths
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e9'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test.csv')
SUBMISSION_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')

# Load data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

## Configs

Define config values to use later.

In [ ]:
# CONFIGS

N_SPLITS = 5
RANDOM_STATE = 42
N_TRIALS = 30

ID = "id"
TARGET = "Will_Buy_EV"

# Splitting Features and Target values
X = train.drop([ID,TARGET],axis=1)
y = train[TARGET]

# For the test data
X_test = test.drop(ID,axis=1)

# Features
FEATURES = X.columns.tolist()

# Categorical Features
cat_features = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level"
]

cat_feature_indices = [
    X.columns.get_loc(col)
    for col in cat_features
]

## Optuna Study

Optimizing the Catboost Model using Optuna hyperparameter tuning.

In [ ]:
# optuna objective function
def objective(trial):

    params = {
        "loss_function": "Logloss",
        "random_seed": 42,
        "verbose": 0,
        "task_type": "GPU",

        "iterations": trial.suggest_int(
            "iterations",
            500,
            3000
        ),

        "depth": trial.suggest_int(
            "depth",
            4,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            20,
            log=True
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            5
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0,
            10
        ),

        "border_count": trial.suggest_int(
            "border_count",
            32,
            255
        )
    }

    early_stopping = trial.suggest_int(
        "early_stopping_rounds",
        50,
        250
    )

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = []

    for train_idx, valid_idx in cv.split(X, y):

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_feature_indices,
            eval_set=(X_valid, y_valid),
            early_stopping_rounds=early_stopping,
            use_best_model=True
        )

        # Probability of positive class
        preds = model.predict_proba(X_valid)[:, 1]

        score = roc_auc_score(
            y_valid,
            preds
        )

        scores.append(score)

        trial.report(
            np.mean(scores),
            step=len(scores)
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)

In [ ]:
# Run Study
study = optuna.create_study(
    direction="maximize",
    study_name="catboost_v1",
    pruner=MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=2
    ),
    storage="sqlite:////kaggle/working/optuna.db",
    load_if_exists=True
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

In [ ]:
# Save best parameters
with open("/kaggle/working/best_catboost_params.json", "w") as f:
    json.dump(study.best_params, f, indent=4)

# Save all optuna trials
trials_df = study.trials_dataframe()

trials_df.to_csv(
    "/kaggle/working/catboost_optuna_trials.csv",
    index=False
)

In [ ]:
# Optimized params by Optuna run
print("Best ROC-AUC:", study.best_value)
print("Best parameters:")
best_params = study.best_params
print(best_params)

In [ ]:
# Final 5 Fold CV
print("Starting 5-Fold Stratified Cross-Validation...")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# OOF predictions
oof_preds = np.zeros(len(X))

# Test predictions
oof_test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\nTraining Fold {fold + 1}/5...")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Create CatBoost model
    model = CatBoostClassifier(
        loss_function="Logloss",
        **best_params,
        random_seed=42,
        verbose=200,
        task_type="GPU"
    )

    # Train
    model.fit(
        X_train,
        y_train,
        cat_features=cat_feature_indices
    )

    # Validation predictions
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds

    # Test predictions
    test_preds = model.predict_proba(X_test)[:, 1]
    oof_test_preds += test_preds / 5

    # Fold ROC-AUC
    fold_auc = roc_auc_score(
        y_val,
        val_preds
    )

    print(f"Fold {fold + 1} ROC-AUC: {fold_auc:.4f}")


print("\nCross-Validation complete!")


# Overall OOF Metrics

# 0.5 threshold for classification metrics
y_pred_binary = (oof_preds >= 0.5).astype(int)

# mapping the targets
y[:] = [1 if x == "Yes" else 0 for x in y]

accuracy = accuracy_score(
    y,
    y_pred_binary
)

precision = precision_score(
    y,
    y_pred_binary
)

recall = recall_score(
    y,
    y_pred_binary
)

f1 = f1_score(
    y,
    y_pred_binary
)

roc_auc = roc_auc_score(
    y,
    oof_preds
)

pr_auc = average_precision_score(
    y,
    oof_preds
)


# Print Overall Metrics
print("\n" + "-" * 40)
print("CatBoost (5-Fold OOF) Performance:")
print("-" * 40)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")

print("-" * 40)

# Classification Report
print("\nClassification Report:")

print(
    classification_report(
        y,
        y_pred_binary
    )
)

# Confusion Matrix
print("Confusion Matrix:")

print(
    confusion_matrix(
        y,
        y_pred_binary
    )
)

## Precision Recall Curve

In [ ]:
# Precision Recall Curve
precision_curve, recall_curve, thresholds = precision_recall_curve(y, oof_preds)

plt.plot(
    recall_curve,
    precision_curve,
    label=f"Catboost (PR-AUC = {pr_auc:.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid()
plt.show()

## Submission

Saving the predicted result for competition submission.

In [ ]:
submission = pd.DataFrame({
    ID: test[ID],
    TARGET: oof_test_preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("Submission saved.")